# **Deep Learning & Generative AI - Project**

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# **Project Overview**

### Description

The dataset for this competition consists of multiple choice questions designed to evaluate reasoning, comprehension, and answer ranking capabilities of AI systems. Each question contains a prompt along with five answer choices labeled A, B, C, D, and E.

Participants are provided with a training dataset containing questions and their corresponding correct answers. The test dataset contains similar questions without the answer column, and participants must predict the top three most probable answers for each question.

The dataset includes a variety of question styles and difficulty levels to encourage models that generalize effectively across different problem types.


### Model

The model is built using the pre-trained Hugging Face transformer architecture (`Qwen/Qwen2.5-0.5B`), specifically adapted for Sequence Classification. The architecture applies a linear classification head on top of the transformer's pooled output to evaluate 5 sequence pairs simultaneously.


# **Environment Setup**

## Standard Imports

In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

## Workflow Reproducibility

In [3]:
# Constants
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Configured Compute Device: {device}")

Configured Compute Device: cuda


In [4]:
# Labels
LABEL_MAP = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
LABEL_LETTERS = ["A", "B", "C", "D", "E"]
REVERSE_LABEL_MAP = {0: "A", 1: "B", 2: "C", 3: "D", 4: "E"}

# **Data Preparation**

## Loading Datasets

In [5]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

In [6]:
print(f"Training instances: {len(train_df)}")
print(f"Testing instances: {len(test_df)}")

Training instances: 2000
Testing instances: 500


## Tokenization (Hugging Face)

Using the standard Hugging Face `AutoTokenizer` to encode the raw text into sub-word token IDs, attention masks, and token type IDs required by the Qwen architecture.


In [7]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
print(f"Loading Tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Loading Tokenizer for Qwen/Qwen2.5-0.5B...


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Hugging Face Dataset

To format the multiple-choice structure for the transformer, the question prompt is repeated 5 times and paired with each of the 5 answer options. The resulting tokenized pairs are stacked together to form a shape of `(5, sequence_length)` per question.


In [8]:
class HFMCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128, is_test=False):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_test = is_test
        self.options = ["A", "B", "C", "D", "E"]
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row["prompt"])
        
        first_sentences = [prompt] * 5
        second_sentences = [str(row[opt]) for opt in self.options]
        
        tokenized = self.tokenizer(
            first_sentences,
            second_sentences,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        item = {key: val for key, val in tokenized.items()}
        
        if not self.is_test:
            item["labels"] = torch.tensor(LABEL_MAP[row["answer"]], dtype=torch.long)
        
        return item

In [9]:
print("Tokenizing full train dataset (this might take a minute)...")
train_dataset = HFMCQDataset(train_df, tokenizer)
test_dataset = HFMCQDataset(test_df, tokenizer, is_test=True)

Tokenizing full train dataset (this might take a minute)...


# **Model Architecture**

The `AutoModelForSequenceClassification` architecture automatically attaches a linear classification head on top of the transformer's pooled representation to evaluate the validity of the 5 question-answer pairs, outputting exactly 5 logits.


## Pre-trained Transformer Initialization

In [10]:
print(f"Loading AutoModelForSequenceClassification for {MODEL_NAME}...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=1, trust_remote_code=True
).to(device)

if model.config.pad_token_id is None:
    model.config.pad_token_id = tokenizer.pad_token_id

Loading AutoModelForSequenceClassification for Qwen/Qwen2.5-0.5B...


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Model Initialization

In [11]:
# Hyperparameters for Pretrained Model
BATCH_SIZE = 8
LR = 1e-5
EPOCHS = 1

# 100% Full Training (No Validation Split)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Hugging Face requires AdamW instead of Adam for optimal weight decay
optimizer = optim.AdamW(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()

# **Training Routine**

The network is optimized using Cross-Entropy Loss (suitable for multi-class classification) and the AdamW optimizer. The training loop computes gradients via backpropagation and updates the weights iteratively.


## Training Loop Execution

In [12]:
print(f"Starting 100% Full Training for Kaggle Submission ({len(train_df)} samples)...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        
        # Flatten inputs for Qwen
        input_ids = batch["input_ids"].view(-1, batch["input_ids"].size(-1))
        attention_mask = batch["attention_mask"].view(-1, batch["attention_mask"].size(-1))
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        
        # Reshape logits back to (batch_size, 5 options)
        logits = outputs.logits.view(-1, 5)
        loss = loss_fn(logits, batch["labels"])
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch["input_ids"].size(0)
        _, predicted = torch.max(logits, 1)
        total += batch["labels"].size(0)
        correct += (predicted == batch["labels"]).sum().item()
        
        if step % 100 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}] Step [{step}/{len(train_loader)}] Loss: {loss.item():.4f}")
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f}")

Starting 100% Full Training for Kaggle Submission (2000 samples)...
Epoch [1/1] Step [0/250] Loss: 1.5469
Epoch [1/1] Step [100/250] Loss: 0.5195
Epoch [1/1] Step [200/250] Loss: 0.2109
Epoch [1/1] | Train Loss: 0.5558 | Train Acc: 0.8295


# **Inference & Submission**

The trained model evaluates the hidden test set. The logits are passed through a Softmax function to obtain probability distributions, and the Top-3 highest probability classes are extracted for the final submission format.


## Test Set Predictions

In [13]:
# Inference on Hidden Test Set
print("\nStarting Inference on Test Set...")
model.eval()

all_predictions = []
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        
        input_ids = batch["input_ids"].view(-1, batch["input_ids"].size(-1))
        attention_mask = batch["attention_mask"].view(-1, batch["attention_mask"].size(-1))
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.view(-1, 5)
        
        probs = torch.nn.functional.softmax(logits, dim=1).to(torch.float32)
        top3_indices = torch.topk(probs, 3, dim=-1).indices
        for idx_array in top3_indices.cpu().numpy():
            letters = [REVERSE_LABEL_MAP[i] for i in idx_array]
            all_predictions.append(" ".join(letters))


Starting Inference on Test Set...


## Submission File

In [14]:
# Generate the MAP@3 ready submission dataframe
submission_df = pd.DataFrame({
    "id": test_df["id"],
    "prediction": all_predictions
})
submission_df.to_csv("submission.csv", index=False)
print("submission.csv successfully generated!")
display(submission_df.head())

submission.csv successfully generated!


,id,prediction
0,1,A D E
1,2,B D C
2,3,B C E
3,4,E A C
4,5,C A D


# **DONE**